# Embeddings

In [1]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import cv2

# ==========================================
# 1. HELPER FUNCTIONS (From your original script)
# ==========================================

def load_transform(transform_path):
    with open(transform_path, 'r') as f:
        return json.load(f)

def transform_polygon_to_pixels(polygon_coords, transform):
    """
    Converts a list of [x, y] world coordinates to pixel coordinates.
    """
    coords = np.array(polygon_coords)
    
    # 1. Shift by Min X
    pixel_x = (coords[:, 0] - transform['min_x']) * transform['pixel_per_unit']
    
    # 2. Shift by Max Y and FLIP (because image Y=0 is top, DXF Y=0 is bottom)
    pixel_y = (transform['max_y'] - coords[:, 1]) * transform['pixel_per_unit']
    
    # 3. Stack and Round
    return np.column_stack((pixel_x, pixel_y)).astype(int)

# ==========================================
# 2. DATAFRAME CREATION LOGIC
# ==========================================

def create_layout_dataframe(geojson_path, transform_path):
    """
    Parses GeoJSON and Transform file to create a Pandas DataFrame.
    """
    # Load external files
    try:
        with open(geojson_path, 'r') as f:
            geojson_data = json.load(f)
        transform = load_transform(transform_path)
    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return pd.DataFrame()

    # Derive DXF filename from the geojson filename (assuming convention)
    # e.g., "chunked_data/Floor plan.geojson" -> "Floor plan.dxf"
    base_name = os.path.splitext(os.path.basename(geojson_path))[0]
    dxf_filename = f"{base_name}.dxf"

    rows = []

    print(f"[INFO] Processing {len(geojson_data['features'])} features...")

    for feature in geojson_data['features']:
        props = feature['properties']
        geom = feature['geometry']

        # Extract IDs
        layout_id = props.get('layout_id')
        chunk_id = props.get('chunk_id')

        # Extract and Transform Geometry
        # Note: GeoJSON polygons are usually [[[x,y], [x,y]...]] (list of rings)
        # We take the first ring (exterior)
        if geom and 'coordinates' in geom:
            raw_coords = geom['coordinates'][0]
            
            # Apply the transformation logic to get pixel coordinates
            pixel_poly = transform_polygon_to_pixels(raw_coords, transform)

            rows.append({
                'dxf_file': dxf_filename,
                'layout_id': layout_id,
                'chunk_id': chunk_id,
                'chunks': pixel_poly  # Storing the numpy array of pixels
            })

    # Create DataFrame
    df = pd.DataFrame(rows)
    return df

# ==========================================
# 3. VISUALIZATION FROM DATAFRAME
# ==========================================
def plot_multiple_chunks(df, image_path, target_layouts, target_chunks):
    """
    Queries the DataFrame and plots multiple specific chunks/layouts on the image.
    Accepts lists for target_layouts and target_chunks.
    """
    
    # 0. NORMALIZE INPUTS TO LISTS
    # This ensures the code works even if you pass a single integer (e.g. 0 instead of [0])
    if not isinstance(target_layouts, (list, tuple, np.ndarray)):
        target_layouts = [target_layouts]
    if not isinstance(target_chunks, (list, tuple, np.ndarray)):
        target_chunks = [target_chunks]

    # 1. QUERY THE DATAFRAME
    # Filter using .isin() to match any value in the provided lists
    subset = df[
        (df['layout_id'].isin(target_layouts)) & 
        (df['chunk_id'].isin(target_chunks))
    ]

    if subset.empty:
        print(f"[WARN] No data found for Layouts {target_layouts}, Chunks {target_chunks}")
        return

    # 2. Load Image
    try:
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except Exception as e:
        print(f"[ERROR] Could not load image: {e}")
        return

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(img)

    # 3. Plot all polygons associated with the query
    pixel_polygons = []
    
    # Generate distinct colors for the number of unique items we found
    # We create a unique key for every (layout, chunk) pair to assign a color
    unique_pairs = subset[['layout_id', 'chunk_id']].drop_duplicates()
    cmap = plt.get_cmap('tab20') # 'tab20' has 20 distinct colors
    
    # Create a dictionary mapping (layout_id, chunk_id) -> Color
    color_map = {}
    for idx, (l_id, c_id) in enumerate(zip(unique_pairs['layout_id'], unique_pairs['chunk_id'])):
        color_map[(l_id, c_id)] = cmap(idx % 20)

    for _, row in subset.iterrows():
        poly_points = row['chunks']
        pixel_polygons.append(poly_points)
        
        # Determine color based on this row's IDs
        current_color = color_map.get((row['layout_id'], row['chunk_id']), 'lime')
        
        patch = MplPolygon(
            poly_points, 
            closed=True, 
            facecolor=current_color, 
            edgecolor='white', 
            alpha=0.6,
            linewidth=2
        )
        ax.add_patch(patch)

    # 4. Zoom camera to the specific chunks
    if pixel_polygons:
        all_points = np.vstack(pixel_polygons)
        min_x, min_y = all_points.min(axis=0)
        max_x, max_y = all_points.max(axis=0)
        
        pad = 50
        ax.set_xlim(min_x - pad, max_x + pad)
        ax.set_ylim(max_y + pad, min_y - pad) # Invert Y for image plotting
    
    ax.set_title(f"Multi-Chunk Query: Layouts {target_layouts} | Chunks {target_chunks}")
    plt.show()

# ==========================================
# 4. EXECUTION
# ==========================================
if __name__ == "__main__":
    
    # File Paths
    GEOJSON_FILE = "chunked_data/冷冻机房0327_t3.geojson"
    TRANSFORM_FILE = "images/冷冻机房0327_t3_transform.json"
    IMAGE_FILE   = "images/冷冻机房0327_t3.png"

    # 1. Create the DataFrame
    df = create_layout_dataframe(GEOJSON_FILE, TRANSFORM_FILE)

[INFO] Processing 401 features...


## Link geometry to images

In [2]:
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt # Needed for colormap
import matplotlib.cm as cm 
import numpy as np

def get_chunk_pillow_images(df, image_path, target_layouts, target_chunks, padding=50):
    """
    Queries the DataFrame, calculates the bounding box of specific chunks,
    and returns TWO Pillow images:
    1. The raw cropped region (clean).
    2. The cropped region with segmentation overlay.

    Returns:
        (PIL.Image, PIL.Image): Tuple (raw_crop, overlay_crop). 
                                Returns (None, None) if no data found.
    """
    
    # 1. NORMALIZE INPUTS
    if not isinstance(target_layouts, (list, tuple, np.ndarray)):
        target_layouts = [target_layouts]
    if not isinstance(target_chunks, (list, tuple, np.ndarray)):
        target_chunks = [target_chunks]

    # 2. QUERY DATAFRAME
    subset = df[
        (df['layout_id'].isin(target_layouts)) & 
        (df['chunk_id'].isin(target_chunks))
    ]

    if subset.empty:
        print(f"[WARN] No data found for Layouts {target_layouts}, Chunks {target_chunks}")
        return None, None

    # 3. LOAD IMAGE
    try:
        # Load as RGBA to handle transparency composition
        base_img = Image.open(image_path).convert("RGBA")
    except Exception as e:
        print(f"[ERROR] Could not load image: {e}")
        return None, None

    # 4. CREATE OVERLAY LAYER
    overlay = Image.new("RGBA", base_img.size, (255, 255, 255, 0))
    draw = ImageDraw.Draw(overlay)

    # Setup Colors
    unique_pairs = subset[['layout_id', 'chunk_id']].drop_duplicates()
    cmap = plt.get_cmap('tab20')
    color_map = {}
    
    for idx, (l_id, c_id) in enumerate(zip(unique_pairs['layout_id'], unique_pairs['chunk_id'])):
        rgba_float = cmap(idx % 20)
        rgb_int = tuple(int(c * 255) for c in rgba_float[:3])
        # Alpha 128 = ~50% transparency
        color_map[(l_id, c_id)] = rgb_int + (128,)

    # 5. DRAW POLYGONS & CALCULATE BOUNDS
    all_points_for_crop = []

    for _, row in subset.iterrows():
        poly_arr = row['chunks']
        poly_tuples = [tuple(pt) for pt in poly_arr]
        
        fill_color = color_map.get((row['layout_id'], row['chunk_id']), (0, 255, 0, 128))
        
        # Draw on the overlay layer
        draw.polygon(poly_tuples, fill=fill_color, outline="white")
        
        all_points_for_crop.append(poly_arr)

    # Create the Combined version
    combined_img = Image.alpha_composite(base_img, overlay)

    # 6. CROP BOTH IMAGES
    if all_points_for_crop:
        all_points = np.vstack(all_points_for_crop)
        
        min_x, min_y = all_points.min(axis=0)
        max_x, max_y = all_points.max(axis=0)

        width, height = base_img.size
        
        # Calculate Box with padding
        left = max(0, min_x - padding)
        top = max(0, min_y - padding)
        right = min(width, max_x + padding)
        bottom = min(height, max_y + padding)
        
        crop_box = (left, top, right, bottom)

        # A. Crop the Raw Base Image (Convert to RGB to drop Alpha channel)
        raw_crop = base_img.crop(crop_box).convert("RGB")
        
        # B. Crop the Combined Image
        overlay_crop = combined_img.crop(crop_box).convert("RGB")
        
        return raw_crop, overlay_crop
    
    # Fallback if no points found (return full images)
    return base_img.convert("RGB"), combined_img.convert("RGB")

# raw_img, overlay_img = get_chunk_pillow_images(
#     df, 
#     IMAGE_FILE, 
#     target_layouts=[0], 
#     target_chunks=[0]
# )

In [3]:
# overlay_img
df

,dxf_file,layout_id,chunk_id,chunks
0,冷冻机房0327_t3.dxf,0,0,"[[1945, 3412], [1926, 3431], [1908, 3412], [19..."
1,冷冻机房0327_t3.dxf,0,0,"[[1502, 3421], [1484, 3403], [1520, 3403], [15..."
2,冷冻机房0327_t3.dxf,0,0,"[[1489, 3177], [1470, 3159], [1507, 3159], [14..."
3,冷冻机房0327_t3.dxf,0,0,"[[1965, 3988], [1970, 3987], [1975, 3987], [19..."
4,冷冻机房0327_t3.dxf,0,0,"[[1298, 3650], [1371, 3650], [1371, 3662], [13..."
...,...,...,...,...
396,冷冻机房0327_t3.dxf,1,9,"[[6858, 3686], [6858, 3662], [7053, 3662], [70..."
397,冷冻机房0327_t3.dxf,1,9,"[[6784, 3686], [6784, 3662], [6858, 3662], [68..."
398,冷冻机房0327_t3.dxf,1,9,"[[6258, 3989], [6263, 3988], [6268, 3987], [62..."
399,冷冻机房0327_t3.dxf,1,9,"[[6580, 3412], [6617, 3412], [6598, 3431], [65..."


## The indexer 

In [ ]:
import os
import random
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import gc
import threading
from transformers import AutoProcessor, AutoModel
from tqdm import tqdm

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
revision_id = "344d954da76eb8ad47a7aaff42d012e30c15b8fe"

model = SentenceTransformer(
    "LocalModels/jina-clip-v2-local", 
    device=device,
    revision=revision_id,
    trust_remote_code=True, 
    truncate_dim=384,
    local_files_only=True
)

INDEX_FILE = 'chunk_embeddings.parquet'

def create_embedding_index(geometry_df, full_image_path):
    print("--- 1. Identify Unique Chunks ---")
    # We only need one row per chunk_id to know it exists and which file it belongs to
    unique_chunks = geometry_df[['layout_id', 'chunk_id', 'dxf_file']].drop_duplicates().reset_index(drop=True)
    
    embeddings = []
    valid_indices = []
    
    print(f"--- 2. Processing {len(unique_chunks)} chunks ---")
    
    for idx, row in tqdm(unique_chunks.iterrows(), total=unique_chunks.shape[0], desc="Encoding"):
        l_id = row['layout_id']
        c_id = row['chunk_id']
        dxf_name = row['dxf_file']
        
        # Get the clean crop
        raw_crop, _ = get_chunk_pillow_images(geometry_df, full_image_path, l_id, c_id)
        
        if raw_crop:
            # Generate Vector
            with torch.no_grad():
                vec = model.encode(raw_crop, show_progress_bar=False)
            embeddings.append(vec)
            valid_indices.append(idx)
        else:
            print(f"Skipping Layout {l_id} Chunk {c_id} (Image load fail)")

    # Filter the unique dataframe to only include successful ones
    final_index_df = unique_chunks.loc[valid_indices].copy()
    final_index_df['embedding'] = embeddings
    
    # SAVE TO PARQUET
    final_index_df.to_parquet(INDEX_FILE)
    print(f"--- Success! Saved {len(final_index_df)} vectors to {INDEX_FILE} ---")

# --- EXECUTE ---
create_embedding_index(df, IMAGE_FILE)

Skipping import of cpp extensions due to incompatible torch version 2.8.0 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0113 13:58:59.529000 57680 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


config.json: 0.00B [00:00, ?B/s]

configuration_xlm_roberta.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- configuration_xlm_roberta.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_lora.py: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


--- 1. Identify Unique Chunks ---
--- 2. Processing 21 chunks ---


Encoding:  67%|██████████████████████           | 14/21 [00:34<00:15,  2.15s/it]

In [ ]:
# import requests
# from PIL import Image
# import torch.nn.functional as F

# image = Image.open(requests.get("https://huggingface.co/datasets/merve/coco/resolve/main/val2017/000000000285.jpg", stream=True).raw)

# with torch.no_grad():
#     image_emb = model.encode(image, show_progress_bar=False, convert_to_tensor=True)

# candidate_labels = ["2 cats", "a plane", "a remote"]
# # text_inputs = processor(text=candidate_labels, padding="max_length", return_tensors="pt").to(model.device)
# with torch.no_grad():
#     # text_emb = model.get_text_features(**text_inputs)
#     text_emb = model.encode(candidate_labels, convert_to_tensor=True)


# # Normalize embeddings for cosine similarity (L2 norm)
# image_emb_norm = F.normalize(image_emb, dim=-1)
# text_emb_norm = F.normalize(text_emb, dim=-1)

# # Compute cosine similarities (dot product of normalized vectors)
# similarities = F.cosine_similarity(image_emb_norm, text_emb_norm, dim=-1)

# # Find the index of the highest similarity
# best_idx = similarities.argmax().item()
# best_label = candidate_labels[best_idx]
# best_score = similarities[best_idx].item()

# print(f"Predicted label: '{best_label}' (score: {best_score:.4f})")
# print("All similarities:", {label: score.item() for label, score in zip(candidate_labels, similarities)})

## The search and display 

In [ ]:
# import os
# import numpy as np
# import pandas as pd
# import torch
# import matplotlib.pyplot as plt
# from sklearn.metrics.pairwise import cosine_similarity
# from PIL import Image

# # Assume these match your environment
# # IMAGE_FOLDER = "./images" 

# def image_retrieval(query_input, layout_filter=None, top_k=3, visualize=True):
#     """
#     query_input: Text string OR PIL Image object
#     visualize: Boolean, set to False to suppress matplotlib output.
#     Returns: A dictionary keyed by layout_id, containing lists of chunk_ids and scores.
#     """
    
#     # Initialize empty result dictionary
#     results = {}

#     # 1. LOAD INDEX
#     if not os.path.exists(INDEX_FILE):
#         print("Index file not found. Run Part 2 first.")
#         return results
        
#     index_df = pd.read_parquet(INDEX_FILE)
    
#     # 2. APPLY LAYOUT FILTER
#     if layout_filter is not None:
#         index_df = index_df[index_df['layout_id'].isin(layout_filter)]
#         if index_df.empty:
#             print("No chunks found in the specified layouts.")
#             return results

#     # 3. PREPARE VECTORS
#     with torch.no_grad():
#         query_vec = model.encode(query_input)
    
#     if isinstance(query_vec, torch.Tensor):
#         query_vec = query_vec.cpu().numpy()
        
#     query_vec = query_vec.reshape(1, -1)
    
#     # Convert database column to matrix
#     db_matrix = np.stack(index_df['embedding'].values) 
    
#     # 4. COMPUTE SIMILARITY
#     scores = cosine_similarity(query_vec, db_matrix)[0]
    
#     # 5. RETRIEVE TOP K
#     top_indices = np.argsort(scores)[::-1][:top_k]
    
#     if visualize:
#         print(f"--- Top {top_k} Matches ---")
#         plt.figure(figsize=(15, 5))
    
#     # 6. PROCESS RESULTS
#     for i, idx in enumerate(top_indices):
#         result_row = index_df.iloc[idx]
#         score = float(scores[idx])
        
#         l_id = int(result_row['layout_id'])
#         c_id = int(result_row['chunk_id'])
        
#         # --- NEW: Organize by Layout ID ---
#         # Ensure the layout key exists
#         if l_id not in results:
#             results[l_id] = {'chunk_ids': [], 'scores': []}
        
#         # Append data to the specific layout
#         results[l_id]['chunk_ids'].append(c_id)
#         results[l_id]['scores'].append(score)
        
#         # 7. OPTIONAL VISUALIZATION
#         if visualize:
#             print(f"Rank {i+1}: Layout {l_id} | Chunk {c_id} | Score: {score:.4f}")
            
#             image_filename = IMAGE_FILE 
            
#             # Re-crop for display
#             img, _ = get_chunk_pillow_images(df, image_filename, l_id, c_id)
            
#             # Plot
#             ax = plt.subplot(1, top_k, i + 1)
#             if img:
#                 ax.imshow(img)
#             else:
#                 ax.text(0.5, 0.5, "Img Not Found", ha='center')
                
#             ax.set_title(f"L:{l_id} C:{c_id}\n{score:.2f}")
#             ax.axis("off")
            
#     if visualize:
#         plt.show()

#     return results

# # --- USAGE ---
# # 1. With visualization (Default)
# # image_retrieval("kitchen", layout_filter=[0, 1], top_k=3)

# # 2. Without visualization (Data only)
# data = image_retrieval("bedroom, master bedroom, master, sleeping zones", top_k=5, visualize=False)
# print(data)

In [ ]:
def sample_indices_per_layout(df, n_samples=10):
    out = {}

    for layout_id, g in df.groupby("layout_id"):
        n = g["chunk_id"].nunique()

        # handle very small layouts safely
        if n <= 1:
            out[layout_id] = np.array([0])
            continue

        idx = (
            np.quantile(np.arange(n), np.linspace(0., 1., n_samples))
            .round()
            .astype(int)
        )

        # ensure valid bounds & uniqueness
        idx = np.unique(np.clip(idx, 0, n - 1))
        out[layout_id] = idx

    return out

# indices_per_layout = sample_indices_per_layout(df, n_samples=int(len(df)*0.2))
# indices_per_layout

In [ ]:
# query_crop, overlay_crop = get_chunk_pillow_images(
#     df, 
#     IMAGE_FILE, 
#     target_layouts=[0], 
#     target_chunks=[1]
# )

In [ ]:
# overlay_crop

# Image search

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

def search_by_image(query_image, layout_filter=None, top_k=3, top_p=None, visualize=True):
    """
    query_image: PIL Image object (e.g., a crop)
    layout_filter: List of layout_ids to restrict search to.
    top_k: Minimum number of results to return.
    top_p: Score threshold (0.0 to 1.0). If scores are higher than this, 
           they are included even if they exceed top_k. 
           Set to None to disable.
    visualize: Boolean, set to False to suppress matplotlib output.
    
    Returns: A dictionary keyed by layout_id, containing lists of chunk_ids and scores.
    """
    
    # Initialize empty result dictionary
    results = {}

    # 1. LOAD INDEX
    if not os.path.exists(INDEX_FILE):
        print("Index file not found. Run Part 2 first.")
        return results
        
    index_df = pd.read_parquet(INDEX_FILE)
    
    # 2. APPLY LAYOUT FILTER
    if layout_filter is not None:
        index_df = index_df[index_df['layout_id'].isin(layout_filter)]
        if index_df.empty:
            print("No chunks found in the specified layouts.")
            return results

    # 3. PREPARE VECTORS (Image specific encoding)
    with torch.no_grad():
        # Encode the image (raw_crop)
        query_vec = model.encode(query_image, show_progress_bar=False)
    
    # Ensure CPU/Numpy format
    if isinstance(query_vec, torch.Tensor):
        query_vec = query_vec.cpu().numpy()
        
    # Reshape 1D -> 2D
    query_vec = query_vec.reshape(1, -1)
    
    # Convert database column to matrix
    db_matrix = np.stack(index_df['embedding'].values) 
    
    # 4. COMPUTE SIMILARITY
    scores = cosine_similarity(query_vec, db_matrix)[0]
    
    # 5. RETRIEVE (HYBRID TOP_K AND TOP_P)
    # Sort all scores descending
    sorted_indices_all = np.argsort(scores)[::-1]
    
    final_indices = []
    
    for idx in sorted_indices_all:
        score = scores[idx]
        
        # Condition 1: Have we met the minimum Top K quota?
        is_within_k = len(final_indices) < top_k
        
        # Condition 2: Does the score satisfy Top P?
        is_above_p = (top_p is not None) and (score >= top_p)
        
        # LOGIC: Keep adding if we are under K OR if the quality is high enough
        if is_within_k or is_above_p:
            final_indices.append(idx)
        else:
            # Since scores are sorted descending, once we fail both checks,
            # no subsequent items will pass.
            break
            
    final_indices = np.array(final_indices)
    num_results = len(final_indices)

    if visualize:
        print(f"--- Found {num_results} Image Matches (k={top_k}, p={top_p}) ---")
        # Adjust figure size dynamically based on number of results
        plt.figure(figsize=(max(15, 3 * num_results), 5))
    
    # 6. PROCESS RESULTS
    for i, idx in enumerate(final_indices):
        result_row = index_df.iloc[idx]
        
        # Cast to standard Python types for cleaner dictionary output
        score = float(scores[idx])
        l_id = int(result_row['layout_id'])
        c_id = int(result_row['chunk_id'])
        
        # --- Organize by Layout ID ---
        if l_id not in results:
            results[l_id] = {'chunk_ids': [], 'scores': []}
        
        results[l_id]['chunk_ids'].append(c_id)
        results[l_id]['scores'].append(score)
        
        # 7. OPTIONAL VISUALIZATION
        if visualize:
            print(f"Rank {i+1}: Layout {l_id} | Chunk {c_id} | Score: {score:.4f}")
            
            image_filename = IMAGE_FILE 
            
            # Re-crop for display
            img, _ = get_chunk_pillow_images(df, image_filename, l_id, c_id)
            
            # Plot (Dynamically size grid based on num_results)
            ax = plt.subplot(1, num_results, i + 1)
            if img:
                ax.imshow(img)
            else:
                ax.text(0.5, 0.5, "Img Not Found", ha='center')
                
            ax.set_title(f"L:{l_id} C:{c_id}\n{score:.2f}")
            ax.axis("off")
            
    if visualize:
        plt.show()

    return results

# # --- USAGE EXAMPLE ---
# data = search_by_image(query_crop, top_k=5, top_p=0.90, visualize=False)
# # print(data)

In [ ]:
# data

In [ ]:
# raw_img, overlay_img = get_chunk_pillow_images(
#     df, 
#     IMAGE_FILE, 
#     target_layouts=[1], 
#     target_chunks=[4, 8, 2]
# )

In [ ]:
# overlay_img

In [ ]:
# query_crop.save("query_crop.png")

In [ ]:
# # main program
# indices_per_layout = sample_indices_per_layout(df, n_samples=int(len(df)*0.2))
# query_crop, overlay_crop = get_chunk_pillow_images(
#     df, 
#     IMAGE_FILE, 
#     target_layouts=[0], 
#     target_chunks=[1]
# )
# data = search_by_image(query_crop, top_k=3, top_p=0.90, visualize=False)

# raw_img, overlay_img = get_chunk_pillow_images(
#     df, 
#     IMAGE_FILE, 
#     target_layouts=[1], 
#     target_chunks=[4, 8, 2]
# )

In [ ]:
# indices_per_layout

In [ ]:
# data

In [ ]:
(4 * 20 * 5) / 60

# Clustering

In [ ]:
# df

In [ ]:
import pandas as pd
import numpy as np
import umap
from sklearn.cluster import DBSCAN, AgglomerativeClustering, HDBSCAN
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def cluster_embeddings(index_file, 
                       method='hdbscan', 
                       umap_n_neighbors=32, 
                       visualize=True,
                       # Alg specific params
                       dbscan_eps=0.5,
                       agg_distance_threshold=1.5): # Lower = Stricter
    """
    Methods: 'dbscan', 'hdbscan', 'agglomerative'
    """
    
    # --- 1. Load & Reduce Data ---
    if not os.path.exists(index_file): return {}
    df = pd.read_parquet(index_file)
    matrix = np.stack(df['embedding'].values)
    
    print(f"Reducing {len(df)} embeddings with UMAP...")
    reducer = umap.UMAP(n_neighbors=umap_n_neighbors, min_dist=0.0, n_components=3, metric='cosine', random_state=42)
    # reducer = PCA(n_components=0.90)
    embedding_2d = reducer.fit_transform(matrix)
    
    # --- 2. Apply Clustering ---
    print(f"Clustering using {method.upper()}...")
    
    if method == 'dbscan':
        clusterer = DBSCAN(eps=dbscan_eps, min_samples=2)
        labels = clusterer.fit_predict(embedding_2d)
        
    elif method == 'hdbscan':
        # min_cluster_size: Smallest grouping to be considered a cluster
        # min_samples: How conservative you want to be (larger = more noise/unclustered)
        clusterer = HDBSCAN(min_cluster_size=3, min_samples=3) 
        labels = clusterer.fit_predict(embedding_2d)
        
    elif method == 'agglomerative':
        # distance_threshold is the linkage distance above which clusters will not be merged.
        # Since UMAP output is not normalized 0-1, you usually need to experiment.
        # Start around 1.0 - 5.0 for UMAP data.
        clusterer = AgglomerativeClustering(
            n_clusters=None, 
            distance_threshold=agg_distance_threshold,
            linkage='ward' # 'ward' minimizes variance (makes compact blobs)
        )
        labels = clusterer.fit_predict(embedding_2d)
        
    else:
        raise ValueError("Unknown method")

    # Save Results
    df['cluster_label'] = labels
    df['umap_x'] = embedding_2d[:, 0]
    df['umap_y'] = embedding_2d[:, 1]
    
    # Stats
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    print(f"Found {n_clusters} clusters.")

    # --- 3. Visualization ---
    if visualize:
        plt.figure(figsize=(10, 6))
        # Plot Noise (-1)
        noise = df[df['cluster_label'] == -1]
        plt.scatter(noise['umap_x'], noise['umap_y'], c='lightgrey', s=15, label='Noise')
        # Plot Clusters
        clustered = df[df['cluster_label'] != -1]
        if not clustered.empty:
            sc = plt.scatter(clustered['umap_x'], clustered['umap_y'], 
                             c=clustered['cluster_label'], cmap='tab20', s=15)
            plt.colorbar(sc, label='Cluster ID')
        plt.title(f"Method: {method.upper()} | Clusters: {n_clusters}")
        plt.show()

    # --- 4. Format Output (Your Standard Format) ---
    formatted_groups = {}
    for cluster_id, group_df in df.groupby('cluster_label'):
        if cluster_id == -1: continue # Skip noise
        
        layout_map = {}
        for _, row in group_df.iterrows():
            l_id = int(row['layout_id'])
            c_id = int(row['chunk_id'])
            if l_id not in layout_map: layout_map[l_id] = {'chunk_ids': [], 'scores': []}
            layout_map[l_id]['chunk_ids'].append(c_id)
            layout_map[l_id]['scores'].append(1.0) # Dummy score for clustering
            
        formatted_groups[int(cluster_id)] = layout_map
        
    return formatted_groups

# --- USAGE EXAMPLES ---

# OPTION A: HDBSCAN (Recommended)
# Best for finding groups without worrying about parameters
groups_hdb = cluster_embeddings(INDEX_FILE, method='agglomerative', agg_distance_threshold=2.0)

# OPTION B: Agglomerative
# Best if you want to enforce a strict "tightness"
# groups_agg = cluster_embeddings(INDEX_FILE, method='agglomerative', agg_distance_threshold=2.0)

In [ ]:
groups_hdb[0]

In [ ]:
# clustered_data[10]

In [ ]:
raw_img, overlay_img = get_chunk_pillow_images(
    df, 
    IMAGE_FILE, 
    target_layouts=[1], 
    target_chunks=[0, 4, 8]
)

In [ ]:
overlay_img

In [ ]:
# groups_hdb

In [ ]:
# raw_img